In [9]:
import os, requests

# 1) Set env vars in THIS Python session
os.environ["APCA_API_KEY_ID"] = "PKIJS7KQYCZX2BDQFCZO6YQ2MO"
os.environ["APCA_API_SECRET_KEY"] = "9ki2UdXi2FfmmxtJ6Ewr22zbud1xTCpsgTvHq86E8Ygr"

# 2) Build headers from env vars (safer than hard-coding later)
BASE = "https://paper-api.alpaca.markets"
H = {
    "APCA-API-KEY-ID": os.getenv("APCA_API_KEY_ID"),
    "APCA-API-SECRET-KEY": os.getenv("APCA_API_SECRET_KEY"),
}

# 3) Quick who-am-I check
r = requests.get(f"{BASE}/v2/account", headers=H, timeout=10)
r.raise_for_status()
acct = r.json()
print("Account ID:", acct.get("id"))
print("Status:", acct.get("status"))
print("Equity:", acct.get("equity"))

Account ID: ee3e7510-0a6f-44e0-9c66-6e7d6b9c17f7
Status: ACTIVE
Equity: 30000


In [10]:
# === Build WATCHLIST from latest infer CSV in a specific folder (with optional Alpaca tradability check) ===
import os, re, time
from pathlib import Path
import pandas as pd
import requests

# ---- set your infer folder here (use r'' for Windows paths) ----
INFER_DIR = Path(r"C:\Users\brobi\OneDrive\Desktop\Algo1\logs")

# Reuse existing Alpaca settings if present
TRADING_BASE = globals().get("TRADING_BASE", "https://paper-api.alpaca.markets")
HEADERS = globals().get("APCA_HEADERS", globals().get("H", None))

def find_latest_infer_csv(search_dir: Path, patterns=("infer_*.csv", "*infer*.csv", "live_snapshot*.csv")) -> Path | None:
    files = []
    for pat in patterns:
        files += list(search_dir.glob(pat))
    if not files:
        return None
    files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return files[0]

def load_infer_symbols(path: Path, filter_buy: bool = False) -> list[str]:
    df = pd.read_csv(path)
    # choose a symbol column case-insensitively
    sym_col = next((c for c in df.columns if str(c).strip().lower() in ("symbol","ticker","sy")), None)
    if not sym_col:
        raise ValueError(f"No symbol/ticker column found in {path.name} (looked for symbol/ticker/sy).")
    if filter_buy and "side" in df.columns:
        df = df[df["side"].astype(str).str.upper().eq("BUY")]
    tickers = df[sym_col].astype(str).str.upper().str.strip()
    tickers = tickers[tickers.str.match(r"^[A-Z][A-Z0-9.\-]*$")].dropna().drop_duplicates().tolist()
    return tickers

def is_tradable_us_equity(symbol: str) -> bool:
    try:
        r = requests.get(f"{TRADING_BASE}/v2/assets/{symbol}", headers=HEADERS, timeout=10)
        if r.status_code != 200:
            return False
        a = r.json()
        cls = (a.get("asset_class") or a.get("class") or "").lower()
        status = str(a.get("status", "")).lower()
        tradable = bool(a.get("tradable"))
        return (cls == "us_equity") and (status == "active") and tradable
    except Exception:
        return False
    return False
infer_path = find_latest_infer_csv(INFER_DIR)
if not infer_path:
    raise FileNotFoundError(f"No infer CSV found in {INFER_DIR}. Expected names like infer_YYYY-MM-DD.csv")

raw_syms = load_infer_symbols(infer_path, filter_buy=False)
print(f"[watchlist] Found {len(raw_syms)} symbols in {infer_path.name}")

APPLY_TRADABLE_FILTER = True
if APPLY_TRADABLE_FILTER:
    good, bad = [], []
    for i, s in enumerate(raw_syms, 1):
        (good if is_tradable_us_equity(s) else bad).append(s)
        if i % 50 == 0:
            time.sleep(0.2)
    WATCHLIST = sorted(good)
    print(f"[watchlist] Kept {len(good)} tradable; dropped {len(bad)} non-tradable (first few dropped): {bad[:10]}")
else:
    WATCHLIST = sorted(raw_syms)

print(f"[watchlist] Final count: {len(WATCHLIST)}")
WATCHLIST[:20]


[watchlist] Found 62 symbols in infer_2025-11-06.csv
[watchlist] Kept 62 tradable; dropped 0 non-tradable (first few dropped): []
[watchlist] Final count: 62


['AAPL',
 'AEP',
 'AI',
 'ALB',
 'AMD',
 'AMSC',
 'ARQQ',
 'ARRY',
 'BHP',
 'BYND',
 'CHGG',
 'CLNE',
 'COST',
 'ELVA',
 'ENPH',
 'ENVX',
 'EOSE',
 'EVGO',
 'FLNC',
 'FSLR']

In [11]:
# --- Glue: unify Alpaca config so all cells use the same headers ---
import os

# Base URLs (paper)
TRADING_BASE = globals().get("TRADING_BASE", "https://paper-api.alpaca.markets")
DATA_BASE    = globals().get("DATA_BASE",    "https://data.alpaca.markets")

# Prefer an existing APCA_HEADERS; else fall back to H; else build from env vars
APCA_HEADERS = (
    globals().get("APCA_HEADERS")
    or globals().get("H")
    or {
        "APCA-API-KEY-ID": os.getenv("APCA_API_KEY_ID"),
        "APCA-API-SECRET-KEY": os.getenv("APCA_API_SECRET_KEY"),
        "Content-Type": "application/json",
    }
)

# Optional: also expose HEADERS for any cells that used that name
HEADERS = APCA_HEADERS
print("Headers ready (paper).")


Headers ready (paper).


In [12]:
# === Build per-symbol expectations from your latest infer file ===
from pathlib import Path
import os, glob, pandas as pd, numpy as np
from datetime import datetime

# ---- config/fallbacks (tweak later as you learn more) ----
INFER_DIR = Path(r"C:\Users\brobi\OneDrive\Desktop\Algo1\logs")
DEFAULTS = {
    "stop_pct": 0.009,            # 0.9% hard stop below entry
    "trail_pct": 0.025,           # trail stop: giveback from peak
    "tp_pct": 0.035,              # take-profit %
    "per_trade_budget": 1200.0,   # USD budget if no pos_frac
    "pos_frac": None              # fraction of deployable equity (optional)
}

def _latest_infer_csv(d: Path) -> Path | None:
    files = list(d.glob("infer_*.csv")) + list(d.glob("*infer*.csv")) + list(d.glob("live_snapshot*.csv"))
    return max(files, key=lambda p: p.stat().st_mtime) if files else None

def _pick_col(df, names):
    # case-insensitive fuzzy pick
    low = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in low:
            return low[n.lower()]
    return None

def build_expectations(infer_path: Path, defaults: dict = DEFAULTS) -> pd.DataFrame:
    raw = pd.read_csv(infer_path)
    # symbol column
    sym_col = _pick_col(raw, ["symbol", "ticker", "sy"])
    if not sym_col:
        raise ValueError("No symbol/ticker/sy column found in infer file.")
    df = raw.copy()

    # optional filter to BUY rows if present
    side_col = _pick_col(df, ["side"])
    if side_col:
        df = df[df[side_col].astype(str).str.upper().eq("BUY")].copy()

    # map possible column names → unified expectations
    col_map = {
        "stop_pct":      _pick_col(df, ["stop_pct","stop_loss_pct","hard_stop_pct","stop_loss","stop"]),
        "trail_pct":     _pick_col(df, ["trail_pct","trail_from_peak_pct","trailing_pct","dd_trail_pct"]),
        "tp_pct":        _pick_col(df, ["tp_pct","take_profit_pct","take_profit","target_pct"]),
        "pos_frac":      _pick_col(df, ["pos_frac","weight","allocation","alloc","position_fraction"]),
        "per_trade_budget": _pick_col(df, ["budget_usd","per_trade_budget","budget"]),
        "priority":      _pick_col(df, ["score","priority","rank","confidence"])
    }

    out = pd.DataFrame({
        "symbol": df[sym_col].astype(str).str.upper().str.strip()
    })
    for k, src in col_map.items():
        if src:
            out[k] = pd.to_numeric(df[src], errors="coerce")
        else:
            out[k] = np.nan

    # fill fallbacks
    out["stop_pct"]  = out["stop_pct"].fillna(defaults["stop_pct"])
    out["trail_pct"] = out["trail_pct"].fillna(defaults["trail_pct"])
    out["tp_pct"]    = out["tp_pct"].fillna(defaults["tp_pct"])

    # prefer pos_frac; else per_trade_budget; else default
    out["pos_frac"] = out["pos_frac"].where(out["pos_frac"].notna(), defaults["pos_frac"])
    out["per_trade_budget"] = out["per_trade_budget"].where(out["per_trade_budget"].notna(), defaults["per_trade_budget"])

    # de-dup by symbol, keep highest priority if present
    if col_map["priority"]:
        out = (out
               .sort_values(["symbol", "priority"], ascending=[True, False])
               .drop_duplicates(subset=["symbol"], keep="first"))
    else:
        out = out.drop_duplicates(subset=["symbol"], keep="first")

    out["infer_file"] = infer_path.name
    out["infer_loaded_at"] = datetime.utcnow().isoformat(timespec="seconds")
    out = out.reset_index(drop=True)
    print(f"[expectations] {len(out)} symbols loaded from {infer_path.name}")
    return out

infer_csv = _latest_infer_csv(INFER_DIR)
if infer_csv is None:
    raise FileNotFoundError(f"No infer CSV found in {INFER_DIR}")
EXPECT = build_expectations(infer_csv)

# Peek
EXPECT.head(10)


[expectations] 26 symbols loaded from infer_2025-11-06.csv


,symbol,stop_pct,trail_pct,tp_pct,pos_frac,per_trade_budget,priority,infer_file,infer_loaded_at
0,AAPL,0.009,0.025,0.035,0.3,1200.0,NaN,infer_2025-11-06.csv,2025-11-07T04:27:24
1,ALB,0.009,0.025,0.035,0.2,1200.0,NaN,infer_2025-11-06.csv,2025-11-07T04:27:24
2,AMD,0.009,0.025,0.035,0.4,1200.0,NaN,infer_2025-11-06.csv,2025-11-07T04:27:24
3,ARQQ,0.009,0.025,0.035,0.2,1200.0,NaN,infer_2025-11-06.csv,2025-11-07T04:27:24
4,BYND,0.009,0.025,0.035,0.4,1200.0,NaN,infer_2025-11-06.csv,2025-11-07T04:27:24
5,CHGG,0.009,0.025,0.035,0.2,1200.0,NaN,infer_2025-11-06.csv,2025-11-07T04:27:24
6,COST,0.009,0.025,0.035,0.4,1200.0,NaN,infer_2025-11-06.csv,2025-11-07T04:27:24
7,ENPH,0.009,0.025,0.035,0.2,1200.0,NaN,infer_2025-11-06.csv,2025-11-07T04:27:24
8,EVGO,0.009,0.025,0.035,0.2,1200.0,NaN,infer_2025-11-06.csv,2025-11-07T04:27:24
9,GEVO,0.009,0.025,0.035,0.4,1200.0,NaN,infer_2025-11-06.csv,2025-11-07T04:27:24


In [13]:
# === Record entries and attach expectations to compute absolute stop/TP levels ===
from pathlib import Path
import pandas as pd
from datetime import datetime

LEDGER_PATH = Path("./out/positions_ledger.csv")
LEDGER_PATH.parent.mkdir(parents=True, exist_ok=True)

# create ledger if missing
if "LEDGER" not in globals():
    if LEDGER_PATH.exists():
        LEDGER = pd.read_csv(LEDGER_PATH)
    else:
        LEDGER = pd.DataFrame(columns=[
            "symbol","entry_time_utc","shares","entry_px","peak_px",
            "stop_pct","trail_pct","tp_pct",
            "hard_stop_px","tp_px","infer_file"
        ])

def _expect_for(symbol: str, expect_df: pd.DataFrame) -> dict:
    r = expect_df.loc[expect_df["symbol"].eq(symbol.upper())]
    if r.empty:
        # fallback to defaults if symbol missing
        return {
            "stop_pct": DEFAULTS["stop_pct"],
            "trail_pct": DEFAULTS["trail_pct"],
            "tp_pct": DEFAULTS["tp_pct"],
            "per_trade_budget": DEFAULTS["per_trade_budget"],
            "infer_file": None
        }
    row = r.iloc[0]
    return {
        "stop_pct": float(row["stop_pct"]),
        "trail_pct": float(row["trail_pct"]),
        "tp_pct": float(row["tp_pct"]),
        "per_trade_budget": float(row["per_trade_budget"]),
        "infer_file": row.get("infer_file", None)
    }

def record_entry(symbol: str, shares: float, entry_px: float, expect_df: pd.DataFrame = None):
    """
    Add/replace current holding for symbol with fresh thresholds from expectations.
    If symbol already exists, we overwrite with the new entry (fresh trade).
    """
    global LEDGER
    symbol = symbol.upper()
    ex = _expect_for(symbol, expect_df if expect_df is not None else EXPECT)

    hard_stop_px = round(entry_px * (1 - ex["stop_pct"]), 4)
    tp_px        = round(entry_px * (1 + ex["tp_pct"]), 4)

    rec = {
        "symbol": symbol,
        "entry_time_utc": datetime.utcnow().isoformat(timespec="seconds"),
        "shares": float(shares),
        "entry_px": float(entry_px),
        "peak_px": float(entry_px),   # initialize peak at entry
        "stop_pct": float(ex["stop_pct"]),
        "trail_pct": float(ex["trail_pct"]),
        "tp_pct": float(ex["tp_pct"]),
        "hard_stop_px": hard_stop_px,
        "tp_px": tp_px,
        "infer_file": ex["infer_file"]
    }

    # remove any existing row for symbol, then append
    LEDGER = LEDGER[~LEDGER["symbol"].eq(symbol)].reset_index(drop=True)
    LEDGER = pd.concat([LEDGER, pd.DataFrame([rec])], ignore_index=True)

    LEDGER.to_csv(LEDGER_PATH, index=False)
    print(f"[entry] {symbol}: shares={shares}, entry={entry_px:.4f}, "
          f"hard_stop={hard_stop_px:.4f} ({ex['stop_pct']*100:.2f}%), "
          f"tp={tp_px:.4f} ({ex['tp_pct']*100:.2f}%), trail={ex['trail_pct']*100:.2f}% from peak")
    return rec

def update_peak(symbol: str, new_price: float):
    """Bump peak and recompute trailing stop *level* (not stored; computed on read)."""
    global LEDGER
    symbol = symbol.upper()
    i = LEDGER.index[LEDGER["symbol"].eq(symbol)]
    if len(i) == 0:
        return
    idx = i[0]
    prev_peak = float(LEDGER.at[idx, "peak_px"])
    if new_price > prev_peak:
        LEDGER.at[idx, "peak_px"] = float(new_price)
        LEDGER.at[idx, "entry_time_utc"] = datetime.utcnow().isoformat(timespec="seconds")  # touch
        LEDGER.to_csv(LEDGER_PATH, index=False)

def current_thresholds(symbol: str) -> dict | None:
    """Return the absolute levels right now: hard_stop_px, trail_stop_px (from peak), and tp_px."""
    r = LEDGER.loc[LEDGER["symbol"].eq(symbol.upper())]
    if r.empty: 
        return None
    row = r.iloc[0]
    entry_px = float(row["entry_px"])
    peak_px  = float(row["peak_px"])
    stop_pct = float(row["stop_pct"])
    trail_pct= float(row["trail_pct"])
    tp_pct   = float(row["tp_pct"])

    hard_stop_px  = round(entry_px * (1 - stop_pct), 4)
    trail_stop_px = round(peak_px  * (1 - trail_pct), 4)
    tp_px         = round(entry_px * (1 + tp_pct), 4)

    return {
        "symbol": row["symbol"],
        "entry_px": entry_px,
        "peak_px": peak_px,
        "hard_stop_px": hard_stop_px,
        "trail_stop_px": trail_stop_px,
        "tp_px": tp_px,
        "stop_pct": stop_pct,
        "trail_pct": trail_pct,
        "tp_pct": tp_pct
    }

LEDGER.tail(3)


,symbol,entry_time_utc,shares,entry_px,peak_px,stop_pct,trail_pct,tp_pct,hard_stop_px,tp_px,infer_file


In [14]:
# Bootstrap the local LEDGER from broker positions and preview thresholds
# (requires: APCA_HEADERS/APCA key setup, LEDGER helpers, EXPECT, record_entry, current_thresholds)

import pandas as pd, requests, numpy as np

def _broker_positions_df():
    r = requests.get(f"{TRADING_BASE}/v2/positions", headers=APCA_HEADERS, timeout=10)
    if r.status_code == 404:
        return pd.DataFrame(columns=["symbol","qty","avg_entry_price"])
    r.raise_for_status()
    js = r.json()
    if isinstance(js, dict):
        js = js.get("positions", [])
    if not js:
        return pd.DataFrame(columns=["symbol","qty","avg_entry_price"])
    df = pd.DataFrame(js)
    df["qty"] = df["qty"].astype(float)
    df["avg_entry_price"] = df["avg_entry_price"].astype(float)
    return df[["symbol","qty","avg_entry_price"]]

def bootstrap_ledger_from_broker(expect_df=EXPECT):
    df = _broker_positions_df()
    if df.empty:
        print("No broker positions found; LEDGER unchanged.")
        return df
    for _, r in df.iterrows():
        record_entry(
            symbol=str(r["symbol"]).upper(),
            shares=float(r["qty"]),
            entry_px=float(r["avg_entry_price"]),
            expect_df=expect_df
        )
    print(f"Bootstrapped {len(df)} positions into LEDGER.")
    return df

_ = bootstrap_ledger_from_broker()

# Optional: quick preview table of current thresholds vs last price
def _last_close(sym: str):
    try:
        url = f"{DATA_BASE}/v2/stocks/{sym}/bars"
        params = {"timeframe":"1Min","limit":1,"feed":"iex"}
        rr = requests.get(url, headers=APCA_HEADERS, params=params, timeout=10)
        rr.raise_for_status()
        bars = rr.json().get("bars", [])
        return float(bars[-1]["c"]) if bars else np.nan
    except Exception:
        return np.nan

if not LEDGER.empty:
    prev = []
    for sym in LEDGER["symbol"]:
        th = current_thresholds(sym)
        px = _last_close(sym)
        prev.append({
            "symbol": sym,
            "last_px": px,
            "entry_px": th["entry_px"],
            "peak_px": th["peak_px"],
            "hard_stop_px": th["hard_stop_px"],
            "trail_stop_px": th["trail_stop_px"],
            "tp_px": th["tp_px"]
        })
    preview_df = pd.DataFrame(prev).sort_values("symbol")
    display(preview_df)
else:
    print("LEDGER is empty (no positions tracked yet).")


No broker positions found; LEDGER unchanged.
LEDGER is empty (no positions tracked yet).


In [ ]:
import asyncio, pandas as pd, requests

def cancel_symbol_orders(symbol: str):
    try:
        r = requests.get(f"{TRADING_BASE}/v2/orders", headers=APCA_HEADERS, params={"status":"open"}, timeout=10)
        r.raise_for_status()
        for o in r.json():
            if o.get("symbol","").upper() == symbol.upper():
                requests.delete(f"{TRADING_BASE}/v2/orders/{o['id']}", headers=APCA_HEADERS, timeout=10)
    except Exception as e:
        print(f"[cancel] {symbol}: {e}")

def submit_order(payload: dict):
    r = requests.post(f"{TRADING_BASE}/v2/orders", headers=APCA_HEADERS, json=payload, timeout=10)
    if r.status_code >= 400:
        print("[order error]", r.status_code, r.text)
    r.raise_for_status()
    return r.json()

def last_close(symbol: str):
    try:
        url = f"{DATA_BASE}/v2/stocks/{symbol}/bars"
        params = {"timeframe":"1Min","limit":1,"feed":"iex"}
        r = requests.get(url, headers=APCA_HEADERS, params=params, timeout=10)
        r.raise_for_status()
        bars = r.json().get("bars", [])
        return float(bars[-1]["c"]) if bars else None
    except Exception:
        return None

def remove_from_ledger(symbol: str):
    global LEDGER
    LEDGER = LEDGER[~LEDGER["symbol"].eq(symbol.upper())].reset_index(drop=True)
    LEDGER.to_csv(LEDGER_PATH, index=False)

async def guard_once():
    """One pass: update peaks, check thresholds, flatten on breach."""
    if LEDGER.empty:
        return
    for _, r in LEDGER.copy().iterrows():
        sym = str(r["symbol"]).upper()
        px = last_close(sym)
        if px is None:
            continue

        # keep trailing peak fresh
        update_peak(sym, px)

        th = current_thresholds(sym)
        if th is None:
            continue

        shares = int(float(r["shares"]))
        hit_hard  = px <= th["hard_stop_px"]
        hit_trail = px <= th["trail_stop_px"]
        hit_tp    = px >= th["tp_px"]

        if hit_hard or hit_trail or hit_tp:
            try:
                cancel_symbol_orders(sym)  # avoid conflicts with existing child orders
                submit_order({
                    "symbol": sym, "qty": shares, "side": "sell",
                    "type": "market", "time_in_force": "day"
                })
                print(f"[EXIT] {sym} @ ~{px:.2f} | hard={hit_hard} trail={hit_trail} tp={hit_tp}")
            except Exception as e:
                print(f"[EXIT error] {sym}: {e}")
            else:
                remove_from_ledger(sym)

async def guard_loop(interval_sec=300):
    while True:
        try:
            await guard_once()
        except Exception as e:
            print("[guard_loop]", e)
        await asyncio.sleep(interval_sec)

# --- Start the loop (run this to begin) ---
guard_task = asyncio.create_task(guard_loop())
print("5-min guardian started. It will flatten positions when any barrier is hit.")

# --- To stop the loop, run this later:
# guard_task.cancel()
# try:
#     await guard_task
# except asyncio.CancelledError:
#     pass
# print("Guardian stopped.")


5-min guardian started. It will flatten positions when any barrier is hit.


[guardian] 20:52:24 checked 0/0 | exits=0 (ledger empty)
[guardian] 20:57:24 checked 0/0 | exits=0 (ledger empty)
[guardian] 21:02:24 checked 0/0 | exits=0 (ledger empty)
[guardian] 21:07:24 checked 0/0 | exits=0 (ledger empty)
[guardian] 21:12:24 checked 0/0 | exits=0 (ledger empty)
[guardian] 21:17:24 checked 0/0 | exits=0 (ledger empty)
[guardian] 21:22:24 checked 0/0 | exits=0 (ledger empty)
[guardian] 21:27:24 checked 0/0 | exits=0 (ledger empty)
[guardian] 21:32:24 checked 0/0 | exits=0 (ledger empty)
[guardian] 21:37:24 checked 0/0 | exits=0 (ledger empty)
[guardian] 21:42:24 checked 0/0 | exits=0 (ledger empty)
[guardian] 21:47:24 checked 0/0 | exits=0 (ledger empty)
[guardian] 21:52:24 checked 0/0 | exits=0 (ledger empty)
[guardian] 21:57:24 checked 0/0 | exits=0 (ledger empty)
[guardian] 22:02:24 checked 0/0 | exits=0 (ledger empty)


In [18]:
import asyncio
from datetime import datetime, timedelta
from dateutil import tz

# assumes: LEDGER, LEDGER_PATH, last_close, update_peak, current_thresholds,
#          cancel_symbol_orders, submit_order, remove_from_ledger exist
# also assumes TZ is already defined; if not, uncomment the next line:
# TZ = tz.gettz("America/Los_Angeles")

async def guard_once(verbose: bool = True):
    """One pass: update peaks, check thresholds, flatten on breach, print summary."""
    ts = datetime.now(TZ).strftime("%H:%M:%S")

    if LEDGER.empty:
        if verbose:
            print(f"[guardian] {ts} checked 0/0 | exits=0 (ledger empty)")
        return

    total = len(LEDGER)
    checked = 0
    exits = []  # e.g., ["AAPL(hard)", "AMD(trail/tp)"]

    for _, r in LEDGER.copy().iterrows():
        sym = str(r["symbol"]).upper()
        checked += 1

        px = last_close(sym)
        if px is None:
            continue

        # keep trailing peak fresh
        update_peak(sym, px)
        th = current_thresholds(sym)
        if th is None:
            continue

        shares = int(float(r["shares"]))
        reasons = []
        if px <= th["hard_stop_px"]:
            reasons.append("hard")
        if px <= th["trail_stop_px"]:
            reasons.append("trail")
        if px >= th["tp_px"]:
            reasons.append("tp")

        if reasons:
            try:
                cancel_symbol_orders(sym)  # avoid conflicts with open child orders
                submit_order({
                    "symbol": sym, "qty": shares, "side": "sell",
                    "type": "market", "time_in_force": "day"
                })
            except Exception as e:
                print(f"[EXIT error] {sym}: {e}")
            else:
                exits.append(f"{sym}({'/'.join(reasons)})")
                remove_from_ledger(sym)

    if verbose:
        if exits:
            shown = ", ".join(exits[:6]) + ("..." if len(exits) > 6 else "")
            print(f"[guardian] {ts} checked {checked}/{total} | exits={len(exits)} -> {shown}")
        else:
            print(f"[guardian] {ts} checked {checked}/{total} | exits=0")

async def _sleep_to_next_5min():
    now = datetime.now(TZ)
    next_min = ((now.minute // 5) + 1) * 5
    next_tick = now.replace(minute=next_min % 60, second=0, microsecond=0)
    if next_min >= 60:
        next_tick += timedelta(hours=1)
    delay = max(5.0, (next_tick - now).total_seconds())
    print(f"[guardian] next check at {next_tick.strftime('%H:%M:%S')} (~{int(delay)}s)")
    await asyncio.sleep(delay)

async def guard_loop():
    print("✅ 5-minute guardian started.")
    while True:
        try:
            await guard_once(verbose=True)
        except Exception as e:
            print("[guard_loop] error:", e)
        await _sleep_to_next_5min()


In [21]:
import asyncio

guard_task.cancel()
try:
    await guard_task
except asyncio.CancelledError:
    pass
print("Guardian stopped ✅")



Guardian stopped ✅


In [ ]:
guard_task = asyncio.create_task(guard_loop())

✅ 5-minute guardian started.
[guardian] 20:50:26 checked 0/0 | exits=0 (ledger empty)
[guardian] next check at 20:55:00 (~273s)
[guardian] 20:54:59 checked 0/0 | exits=0 (ledger empty)
[guardian] next check at 20:55:00 (~5s)
[guardian] 20:55:05 checked 0/0 | exits=0 (ledger empty)
[guardian] next check at 21:00:00 (~294s)
[guardian] 21:00:00 checked 0/0 | exits=0 (ledger empty)
[guardian] next check at 21:05:00 (~299s)
[guardian] 21:05:00 checked 0/0 | exits=0 (ledger empty)
[guardian] next check at 21:10:00 (~299s)
[guardian] 21:10:00 checked 0/0 | exits=0 (ledger empty)
[guardian] next check at 21:15:00 (~299s)
[guardian] 21:15:00 checked 0/0 | exits=0 (ledger empty)
[guardian] next check at 21:20:00 (~299s)
[guardian] 21:19:59 checked 0/0 | exits=0 (ledger empty)
[guardian] next check at 21:20:00 (~5s)
[guardian] 21:20:05 checked 0/0 | exits=0 (ledger empty)
[guardian] next check at 21:25:00 (~294s)
[guardian] 21:25:00 checked 0/0 | exits=0 (ledger empty)
[guardian] next check at 21

In [23]:
import asyncio

guard_task.cancel()
try:
    await guard_task
except asyncio.CancelledError:
    pass
print("Guardian stopped ✅")

Guardian stopped ✅
